# 状态与会话

## 先把历史显示出来

这一节要讲的是会话。在讲会话之前，先把已经存进数据库、也能通过 `/api/history` 查到的历史记录显示在页面上。

### 前端直接替换

前端不是这一节的重点，直接从 GitHub 的 demo 仓库复制代码：

~~~bash
git clone https://github.com/joylibo/zero-to-tech-demos.git

cp zero-to-tech-demos/zero-to-tech-6-6/components/*.jsx \
  ~/zero-to-tech/components/

cp zero-to-tech-demos/zero-to-tech-6-6/css/lab.css \
  ~/zero-to-tech/css/
~~~

这次新增一个文件、更新三个文件：

- 新增 `HistoryModal.jsx`：历史记录弹窗。
- 更新 `ResultCard.jsx`：右上角增加“历史记录”按钮。
- 更新 `TextLabView.jsx`：控制弹窗开关，点开时才请求 `/api/history`。
- 更新 `lab.css`：按钮和弹窗的样式。

这些前端代码直接搬过来即可，本节不展开。

### 跑起来

启动前端：

~~~bash
cd ~/zero-to-tech
npm run dev
~~~

启动后端：

~~~bash
cd ~/zero-to-tech/backend
source .venv/bin/activate
fastapi dev
~~~

前后端都启动后，打开文字实验室。结果卡右上角会出现“历史记录”按钮，点击后通过 `/api/history` 加载历史记录。

随便分析两句话，再打开历史记录，刚才分析的内容已经显示在弹窗中。

## Cookie：浏览器自动携带的“小纸条”

HTTP 中有一对与 Cookie 有关的头：

| 位置 | HTTP 头 | 含义 |
| --- | --- | --- |
| 响应头 | `Set-Cookie` | 服务端给浏览器一张“小纸条”，下次请求记得带上 |
| 请求头 | `Cookie` | 浏览器把保存的“小纸条”带回服务端 |

流程是：

~~~text
第一次请求
  → 服务端通过 Set-Cookie 发出标识
  → 浏览器保存

后续请求
  → 浏览器自动把标识放进 Cookie 请求头
  → 服务端认出同一个会话
~~~

Cookie 真正重要的地方不是“能在浏览器中存数据”，而是浏览器会按照规则自动携带它。

如果标识由服务器生成，既唯一又难以猜测，再由 Cookie 自动带到每次请求中，就满足了会话标识的三个条件。

## 动手之前，先想清楚三件事

项目要做的事情只有三句：

- 用户第一次到来时，生成一个会话 ID。
- 把状态数据和这个 ID 一起存进数据库。
- 通过 Cookie，让 ID 在浏览器和服务端之间传递。

### 一、会话 ID 怎么生成？

会话 ID 必须唯一且难以猜测，可以使用 UUID（Universally Unique Identifier，通用唯一识别码）

Python 标准库自带 `uuid` 模块，无需安装第三方包。

### 二、Cookie 应该怎么写？

真正发送的 Cookie 类似：

~~~http
session_id=3f8a1c9e42d7460b8e5f1a2c7d9b0e64; HttpOnly; Max-Age=2592000; Path=/; SameSite=lax
~~~

它分成两类：

- `session_id=...` 是 Cookie 的名字和值，也是以后送回服务器的内容。
- 后面的内容是 Cookie 属性，是浏览器需要遵守的设置。

| 属性 | 作用 |
| --- | --- |
| `Max-Age=2592000` | 有效期 30 天，单位是秒；不写时通常是浏览器关闭后消失的临时 Cookie |
| `HttpOnly` | 不允许页面 JavaScript 读取这枚 Cookie |
| `SameSite=lax` | 对跨站携带 Cookie 进行限制 |
| `Path=/` | 当前站点所有路径都可以携带；FastAPI 默认就是 `/` |

FastAPI 提供 `response.set_cookie()`，不需要手写响应头。

### 三、跨源这道坎怎么过？

浏览器有一条安全规则：跨源请求默认不带 Cookie。

当前前端运行在 `localhost:3000`，后端运行在 `localhost:8000`，端口不同，所以是跨源请求。

Cookie 经常充当身份凭证。浏览器要求携带凭证时前后端两边都同意：

- 后端在 CORS 配置中允许携带凭证。
- 前端发请求时明确允许浏览器携带凭证。

两边都设置后，Cookie 才能通过跨源请求传递。

## 开始改造：第〇步，允许跨源请求携带 Cookie

后端在 `CORSMiddleware` 中增加 `allow_credentials=True`：

~~~python
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000"],
    allow_methods=["GET", "POST"],
    allow_credentials=True,
)
~~~

开启 `allow_credentials=True` 后，`allow_origins` 不能写通配符 `"*"`，必须明确写出允许的源。

前端的 InputCard 和 TextLabView 的 `fetch` 都需要加入 `credentials: "include"`。

分析请求：

~~~javascript
const res = await fetch(`${API}/api/analyze`, {
  method: "POST",
  headers: { "Content-Type": "application/json" },
  credentials: "include",
  body: JSON.stringify({ text }),
});
~~~

打开历史记录时的请求：

~~~javascript
async function openHistory() {
  setHistoryOpen(true);
  const res = await fetch(`${API}/api/history`, {
    credentials: "include",
  });
  setHistory(await res.json());
}
~~~

`/api/profile` 不需要 Cookie，因为它返回固定的公开资料，不需要识别访客。

`credentials: "include"` 不是前端手动读取并添加 Cookie，而是允许浏览器自动携带。跨源请求需要这个开关；同源请求中 `fetch` 默认使用 `same-origin`，浏览器会自动携带同源 Cookie。

## 第一步：给表增加 `session_id`

修改 `storage.py` 中的 `init_db()`：

~~~python
def init_db():
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            session_id TEXT,
            text TEXT,
            score REAL,
            label TEXT,
            pinyin TEXT,
            created_at TEXT
        )
    """)
    cur.execute(
        "CREATE INDEX IF NOT EXISTS idx_history_session_created "
        "ON history(session_id, created_at)"
    )
    conn.commit()
    conn.close()
~~~

旧表没有 `session_id` 列，而 `CREATE TABLE IF NOT EXISTS` 不会修改已经存在的表。课程项目的数据还不重要，所以直接删除：

~~~text
backend/history.db
~~~

重启后端后，程序会按照新结构重新创建数据库、表和索引。

## 第二步：写一个“发纸条、认纸条”的工具

这个工具属于接口层，写在 `main.py` 中。顶部导入 `uuid`、`Request` 和 `Response`：

~~~python
import uuid
from fastapi import FastAPI, Request, Response
~~~

新增函数：

~~~python
def get_session_id(request: Request, response: Response) -> str:
    sid = request.cookies.get("session_id")

    if not sid:
        sid = uuid.uuid4().hex
        response.set_cookie(
            "session_id",
            sid,
            httponly=True,
            samesite="lax",
            max_age=60 * 60 * 24 * 30,
        )

    return sid
~~~

逻辑是：

1. 先从请求 Cookie 中读取 `session_id`。
2. 如果没有，说明是第一次来，使用 `uuid.uuid4().hex` 生成新 ID。
3. 使用 `response.set_cookie()` 把 ID 交给浏览器，保存 30 天。
4. 返回当前会话 ID。

`Path=/` 没有显式填写，因为 `set_cookie` 的 `path` 默认值就是 `/`。

## 第三步：存和查都识别 `session_id`

修改 `storage.py`。`save_record` 增加 `session_id` 参数，保存时把会话标识写进记录：

~~~python
def save_record(session_id, record):
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO history "
        "(session_id, text, score, label, pinyin, created_at) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        [
            session_id,
            record["text"],
            record["score"],
            record["label"],
            record["pinyin"],
            record["created_at"],
        ],
    )
    conn.commit()
    conn.close()
~~~

`get_history` 同样增加 `session_id`，只查询这个会话的历史：

~~~python
def get_history(session_id, limit):
    conn = get_conn()
    cur = conn.cursor()
    rows = cur.execute(
        "SELECT * FROM history "
        "WHERE session_id = ? "
        "ORDER BY created_at DESC "
        "LIMIT ?",
        [session_id, limit],
    ).fetchall()
    conn.close()

    records = []
    for row in rows:
        records.append(dict(row))
    return records
~~~

## 第四步：两个接口先认人，再工作

修改 `main.py` 中的两个接口。

分析接口先获得 `sid`，保存时把它交给存储层：

~~~python
@app.post("/api/analyze")
def analyze(
    req: AnalyzeRequest,
    request: Request,
    response: Response,
):
    sid = get_session_id(request, response)

    text = req.text
    score = round(SnowNLP(text).sentiments, 2)
    result = {
        "text": text,
        "score": score,
        "label": score_label(score),
        "pinyin": " ".join(
            lazy_pinyin(text, style=Style.TONE)
        ),
        "created_at": datetime.now(timezone.utc).isoformat(
            timespec="seconds"
        ),
    }

    save_record(sid, result)
    return result
~~~

返回体一个字都没有改变，`session_id` 只通过 Cookie 传递。

历史接口也先获得 `sid`，只返回当前会话的记录：

~~~python
@app.get("/api/history")
def history(
    request: Request,
    response: Response,
    limit: int = 10,
):
    sid = get_session_id(request, response)
    return get_history(sid, limit)
~~~

这次 `limit` 又向外移动了一步。上一节把“一次取几条”的决定从存储层交给 `main.py`；现在调用方可以通过查询参数决定数量：

~~~text
http://localhost:8000/api/history?limit=2
~~~

带 `?limit=2` 时返回两条；不传 `limit` 时使用默认值 10。

## 先看一眼那张“小纸条”

使用 `curl -i`，把响应头和响应体一起显示：

~~~bash
curl -i http://localhost:8000/api/history
~~~

响应头中会出现：

~~~http
set-cookie: session_id=3f8a1c9e42d7460b8e5f1a2c7d9b0e64; HttpOnly; Max-Age=2592000; Path=/; SameSite=lax
~~~

Cookie 本质上就是 HTTP 头中的一行字符串。

`curl` 默认不会保存 Cookie。连续执行两次命令，两次得到的 `session_id` 会不同：第二次请求没有带回第一次收到的纸条，服务端只能把它当成新访客，再生成一枚 Cookie。

浏览器则会替我们保存 Cookie，并在后续请求中自动携带。

### 为什么请求只带回 `session_id`？

服务端通过 `Set-Cookie` 发出的内容包括：

~~~http
session_id=...; HttpOnly; Max-Age=2592000; Path=/; SameSite=lax
~~~

浏览器在请求中送回的只有：

~~~http
Cookie: session_id=...
~~~

原因是 `session_id=...` 是 Cookie 的内容，后面的 `HttpOnly`、`Max-Age`、`Path` 和 `SameSite` 是写给浏览器的属性。

属性由浏览器保存并执行，不需要再报告给服务器。`Cookie` 请求头只携带“名字=值”。

## 边界：会话不是认证

做到这一步，只是使用 Cookie 实现了会话，仍然不能识别访客真实身份。

换一台电脑或者清除 Cookie 后，纸条消失，服务器会把浏览器当成新访客。原来的历史记录并没有从数据库删除，只是浏览器不再拥有对应的 `session_id`，因此无法把记录取出来。

会话维护的是“同一个浏览器的一段连续交互”，不是安全的用户身份。

真正的登录与认证还需要证明“我就是我”，涉及注册、密码安全、验证码、权限和 OAuth 等内容。认证体系可以建立在会话机制之上，但两者不能混为一谈。